# Etapa 5g — Normalización del fondo: resta, división o luminosidad y contraste

**Ronda de mejoras para la entrega final (desvío 14.11 del plan).** Solo desarrollo: las 30 imágenes, la misma validación cruzada por personas de `05f`. El conjunto de prueba no se usa.

**Motivo.** La corrección del fondo del algoritmo congelado **resta** un fondo gaussiano: supone que la iluminación es aditiva. El modelo físico de la imagen (verde = iluminación × reflectancia) es multiplicativo, y una verificación en 10 imágenes de desarrollo lo apoya: en 4.000 puntos de vasos medianos (ancho marcado de 7 a 11 píxeles), la profundidad del vaso crece en proporción al brillo del fondo de al lado (profundidad ≈ 0,24 × fondo; ordenada al origen casi nula; correlación mediana 0,63 por imagen) y el contraste relativo varía menos que el absoluto en 9 de 10 imágenes. Con la resta, un vaso de la periferia oscura queda con la mitad del contraste que uno del centro.

**Candidatas** (`preprocess.normalizar_fondo`, misma escala σ = 0,02 del diámetro), en orden de simplicidad:
- **A. Resta** (el algoritmo congelado): mapa − fondo.
- **B. División:** (mapa − fondo) / |fondo| = 1 − verde / fondo, el contraste relativo.
- **C. Luminosidad y contraste** (Foracchia, Grisan y Ruggeri, 2005): (mapa − L) / C, con la media L y el desvío C locales estimados solo con píxeles de fondo (los que están a menos de un desvío de la media local en una primera pasada). Corrige además el contraste. Se usan ventanas gaussianas en lugar de la grilla del artículo.

**Todo lo demás igual:** mitad de resolución, los 26 mapas calculados sobre el verde normalizado y estandarizados en cada foto, regresión logística con entrenamiento equilibrado por grosor, umbral de **valor fijo** elegido en entrenamiento (la regla del algoritmo final), 5 grupos de personas.

**Medidas**, en las personas de afuera, a resolución original y con todos los píxeles del campo de visión de la base: Matthews, Dice de línea central y sensibilidad de línea central en vasos finos. Descriptivo: Matthews en la periferia (el 25 % exterior del radio del campo) y en el centro, por grupo, y la referencia clásica (detector de líneas solo, con su valor fijo) con cada normalización.

**Regla de adopción (fijada antes de correr).** Una candidata reemplaza a A si
1. no es inferior a A en Matthews ni en Dice de línea central (límite inferior del intervalo al 95 % de la diferencia > −0,03, desvío 14.4), **y**
2. mejora a A en Matthews **o** en vasos finos con un intervalo al 95 % que no incluye el cero.

Si cumplen las dos, gana la más simple (B) salvo que C sea mejor que B en Matthews con un intervalo que no incluye el cero. Si ninguna cumple, queda A.

**Control:** A tiene que reproducir el Matthews de `05d` y `05f` con valor fijo (0,7302).

In [1]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config as C
from src.data import dev_ids, load_split
from src.evaluation import map_images
from src.pipeline import choose_value, fit_weights
from src.split import person_folds
from src.stats import percentile_interval, person_bootstrap_means

pd.set_option("display.width", 220)
ids = dev_ids()
dev = load_split().set_index("image").loc[ids]
folds = dev["person"].map(person_folds(dev.reset_index(), 5, C.SEED))
CANDS = {"A. resta": "resta", "B. división": "división", "C. luminosidad y contraste": "luminosidad y contraste"}
ORDER = list(CANDS)
WORKERS = 4

## Pasada 1: píxeles de entrenamiento y de elección del umbral, con cada normalización

In [2]:
def pass1(image_id):
    from src.pipeline import training_pixels
    return image_id, {c: training_pixels(image_id, background=m) for c, m in CANDS.items()}

t = time.time()
P1 = dict(map_images(pass1, ids, workers=WORKERS))
samples = {c: {i: P1[i][c] for i in ids} for c in CANDS}
del P1
print(f"Pasada 1: {time.time() - t:.0f} s")

Pasada 1: 426 s


## Ajuste por vuelta: pesos y valor fijo con las personas de entrenamiento (y el valor fijo de la referencia)

In [3]:
fit = {}
for k in range(5):
    tr = list(folds.index[folds != k])
    for c in CANDS:
        w = fit_weights(samples[c], tr)
        fit[(k, c)] = (w, choose_value(samples[c], w, tr), choose_value(samples[c], None, tr))
print("Valor fijo elegido en cada vuelta (combinación):")
pd.DataFrame({c: [fit[(k, c)][1] for k in range(5)] for c in CANDS}, index=[f"vuelta {k}" for k in range(5)]).round(4)

Valor fijo elegido en cada vuelta (combinación):


,A. resta,B. división,C. luminosidad y contraste
vuelta 0,0.8713,0.8768,0.8380
vuelta 1,0.8566,0.8739,0.8382
vuelta 2,0.8567,0.8761,0.8367
vuelta 3,0.8567,0.8616,0.8376
vuelta 4,0.8617,0.8697,0.8318


## Pasada 2: máscaras de las personas de afuera, a resolución original

In [4]:
def pass2(image_id):
    import numpy as np
    from scipy import ndimage as ndi
    from src.config import SEGMENT_REFERENCE_FILTER, WIDTH_LIMITS_PX
    from src.data import load_fov, load_image, load_vessels
    from src.enhance import upscale
    from src.fov import detect_fov_robust
    from src.metrics import centerline_dice, matthews
    from src.pipeline import feature_scores
    from src.vessels import skeleton_widths
    k = int(folds[image_id])
    rgb = load_image(image_id)
    truth, fov = load_vessels(image_id), load_fov(image_id)
    auto = detect_fov_robust(rgb)
    skel, wd = skeleton_widths(truth & fov)
    thin = skel & (wd <= WIDTH_LIMITS_PX[0])
    d = ndi.distance_transform_edt(fov)
    peri = fov & (d < 0.25 * d.max())
    rows = []
    for c, mode in CANDS.items():
        w, value, ref_value = fit[(k, c)]
        (p,), kept, _ = feature_scores(rgb, [w], 2, keep=(SEGMENT_REFERENCE_FILTER,), background=mode)
        p = upscale(p, fov.shape)
        r = upscale(kept[SEGMENT_REFERENCE_FILTER], fov.shape)
        m = (p > value) & auto
        mr = (r > ref_value) & auto
        rows.append({"image": image_id, "candidata": c, "matthews": matthews(m, truth, fov),
                     "dice_linea_central": centerline_dice(m, truth, fov), "finos": m[thin].mean(),
                     "matthews_periferia": matthews(m, truth, peri), "matthews_centro": matthews(m, truth, fov & ~peri),
                     "referencia": matthews(mr, truth, fov)})
    return rows

t = time.time()
R = pd.DataFrame([r for rows in map_images(pass2, ids, workers=WORKERS) for r in rows])
R.to_csv(ROOT / "results" / "05g_normalizacion.csv", index=False)
print(f"Pasada 2: {time.time() - t:.0f} s")

Pasada 2: 370 s


## Resultados

In [5]:
def compare(measures):
    out = []
    for meas in measures:
        w = R.pivot(index="image", columns="candidata", values=meas)[ORDER]
        d = w.sub(w["A. resta"], axis=0)
        lo, hi = percentile_interval(person_bootstrap_means(d, dev["person"]))
        for i, c in enumerate(ORDER):
            out.append({"medida": meas, "candidata": c, "media": w[c].mean(), "vs A": d[c].mean(),
                        "inferior": lo[i], "superior": hi[i]})
    return pd.DataFrame(out).set_index(["medida", "candidata"])

table = compare(["matthews", "dice_linea_central", "finos"])
table.round(4)

media    vs A  inferior  superior
medida             candidata                                                     
matthews           A. resta                    0.7302  0.0000    0.0000    0.0000
                   B. división                 0.7509  0.0207    0.0155    0.0266
                   C. luminosidad y contraste  0.6977 -0.0325   -0.0397   -0.0251
dice_linea_central A. resta                    0.7383  0.0000    0.0000    0.0000
                   B. división                 0.7557  0.0174    0.0112    0.0241
                   C. luminosidad y contraste  0.7214 -0.0169   -0.0235   -0.0112
finos              A. resta                    0.3569  0.0000    0.0000    0.0000
                   B. división                 0.3522 -0.0047   -0.0203    0.0133
                   C. luminosidad y contraste  0.4084  0.0515    0.0282    0.0745

In [6]:
print("Descriptivo: periferia (25 % exterior del radio), centro y referencia clásica con cada normalización")
compare(["matthews_periferia", "matthews_centro", "referencia"]).round(4)

Descriptivo: periferia (25 % exterior del radio), centro y referencia clásica con cada normalización


media    vs A  inferior  superior
medida             candidata                                                     
matthews_periferia A. resta                    0.6894  0.0000    0.0000    0.0000
                   B. división                 0.7365  0.0471    0.0308    0.0651
                   C. luminosidad y contraste  0.7087  0.0193    0.0059    0.0343
matthews_centro    A. resta                    0.7433  0.0000    0.0000    0.0000
                   B. división                 0.7549  0.0116    0.0075    0.0163
                   C. luminosidad y contraste  0.6938 -0.0496   -0.0562   -0.0436
referencia         A. resta                    0.7040  0.0000    0.0000    0.0000
                   B. división                 0.7322  0.0282    0.0216    0.0355
                   C. luminosidad y contraste  0.6492 -0.0548   -0.0656   -0.0438

In [7]:
ok = {}
for c in ORDER[1:]:
    noninf = (table.loc[("matthews", c), "inferior"] > -C.NONINF_MARGIN_MATTHEWS
              and table.loc[("dice_linea_central", c), "inferior"] > -C.NONINF_MARGIN_MATTHEWS)
    better = table.loc[("matthews", c), "inferior"] > 0 or table.loc[("finos", c), "inferior"] > 0
    ok[c] = bool(noninf and better)
print("Cumple la regla:", ok)
chosen = "A. resta"
if ok["B. división"] and ok["C. luminosidad y contraste"]:
    w = R.pivot(index="image", columns="candidata", values="matthews")
    d = (w["C. luminosidad y contraste"] - w["B. división"]).to_frame()
    lo, _ = percentile_interval(person_bootstrap_means(d, dev["person"]))
    chosen = "C. luminosidad y contraste" if lo[0] > 0 else "B. división"
elif ok["B. división"]:
    chosen = "B. división"
elif ok["C. luminosidad y contraste"]:
    chosen = "C. luminosidad y contraste"
print("Se adopta:", chosen)

Cumple la regla: {'B. división': True, 'C. luminosidad y contraste': False}
Se adopta: B. división


In [8]:
print("Por grupo (media):")
R.join(dev["group"], on="image").pivot_table(index="candidata", columns="group",
                                              values=["matthews", "finos"]).loc[ORDER].round(3)

Por grupo (media):


finos               matthews              
group                          dr      g      h       dr      g      h
candidata                                                             
A. resta                    0.386  0.355  0.330    0.679  0.741  0.770
B. división                 0.369  0.339  0.348    0.702  0.755  0.796
C. luminosidad y contraste  0.407  0.402  0.416    0.639  0.703  0.751

## Decisión (escrita después de correr)

**Control:** A reproduce exacto el Matthews de `05d` y `05f` con valor fijo (0,7302).

**Se adopta B, la división** (`config.SEGMENT_BACKGROUND_MODE = "división"`). Cumple la regla: no es inferior y mejora Matthews con un intervalo que no incluye el cero.

| Medida (personas de afuera) | A. resta | B. división | B − A (intervalo 95 %) |
|---|---|---|---|
| Matthews | 0,7302 | **0,7509** | **+0,0207** (+0,0155 a +0,0266) |
| Dice de línea central | 0,7383 | 0,7557 | +0,0174 (+0,0112 a +0,0241) |
| Vasos finos | 35,7 % | 35,2 % | −0,5 puntos (−2,0 a +1,3): sin diferencia demostrada |

- **Es consistente:** mejora Matthews en 29 de las 30 imágenes (la que empeora baja 0,0003) y en las 18 personas, en los tres grupos (retinopatía 0,679 → 0,702; glaucoma 0,741 → 0,755; sanos 0,770 → 0,796) y en los dos niveles de compresión (+0,022 y +0,019). El valor fijo elegido por vuelta es estable (0,862 a 0,877).
- **La ganancia está donde la predice el modelo multiplicativo:** en la periferia (25 % exterior del radio) Matthews sube +0,047 (+0,031 a +0,065); en el centro, +0,012. La resta dejaba a los vasos de la periferia oscura con menos contraste que a los del centro; la división los iguala.
- **C (luminosidad y contraste) no cumple:** encuentra más vasos finos (+5,2 puntos) pero es inferior en Matthews (−0,033; −0,040 a −0,025) y empeora el Dice de línea central. Al dividir por el desvío local también amplifica la textura y el ruido de las zonas de poco contraste, como el centro (Matthews en el centro −0,050), y aparecen falsas detecciones.
- **La referencia clásica también mejora con la división** (+0,028), así que la ventaja de la combinación sobre la referencia se achica de 0,026 a 0,019. Se informa: la pregunta principal compara las dos con la misma normalización.

**Cuidados.** La elección se hizo con las mismas 30 imágenes que las rondas anteriores; la validación cruzada cubre los pesos y el umbral, no la elección entre las tres normalizaciones, así que el valor de desarrollo es algo optimista. El conjunto de prueba no se usó.

**Qué cambia en el código.** `SEGMENT_BACKGROUND_MODE` pasa a "división": es el modo con que se calculan los mapas para entrenar y evaluar en desarrollo. Los pesos congelados (`models/pesos_combinacion.json`, etiqueta `congelado-test`) se ajustaron con la resta; ahora el archivo de pesos guarda el modo (`save_weights`) y `segment_final` usa el guardado (`load_background`; si no está, la resta), así que el algoritmo congelado sigue dando exactamente lo mismo. Los pesos nuevos se ajustan con las 30 imágenes al cerrar la ronda de mejoras. Los cuadernos 05d y 08 usan pesos ajustados con la resta: si se vuelven a correr, hay que pasarles `background="resta"` o reajustar.